# 🧠 CBT Conversational AI – LLM Fine-Tuning

This notebook builds a conversational AI model for CBT-style dialogue.

## Goals
- Clean and unify multiple dialogue datasets
- Convert data into chat format (system / user / assistant)
- Fine-tune a lightweight LLM using LoRA
- Prepare the model for real-world conversational use

## Tech Stack
- Hugging Face Transformers
- TRL (SFTTrainer)
- PEFT (LoRA)
- Google Colab (A100 GPU)

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl gradio

In [ ]:
import os
import json
import torch
import gradio as gr

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import SFTTrainer

In [ ]:
print(os.listdir("/content"))

In [ ]:
import json

file_path = "/content/llm_chat_dataset (1).json"

with open(file_path, "r", encoding="utf-8") as f:
    chat_data = json.load(f)

print("Nombre total d'exemples :", len(chat_data))
print(chat_data[0])

In [ ]:
dataset = Dataset.from_list(chat_data)

print(dataset)
print(dataset[0])

assert "messages" in dataset.column_names, "Le dataset doit contenir une colonne 'messages'"

In [ ]:
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

train_ds = dataset_split["train"]
val_ds = dataset_split["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
print(train_ds[0])

In [ ]:
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

train_ds = dataset_split["train"]
val_ds = dataset_split["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

## Load Base Model & Tokenizer

### Objective
Load a lightweight instruction-tuned model.

### Model
Qwen2.5-0.5B-Instruct

### Why this model
- Small → fits Colab GPU
- Fast training
- Good conversational ability

### Setup
- Load tokenizer
- Load model in float16
- Enable automatic device mapping

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded successfully")

In [ ]:
sample_messages = [
    {"role": "system", "content": "Tu es un assistant empathique en français."},
    {"role": "user", "content": "Je me sens stressé ces derniers jours."}
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

print(peft_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen05b-cbt-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_steps=1000,
    save_steps=1000,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
    dataloader_pin_memory=False
)

print(training_args)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer
)

print("Trainer ready")

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("./qwen05b-cbt-lora-adapter")
tokenizer.save_pretrained("./qwen05b-cbt-lora-adapter")

print("Adapter saved")

In [ ]:
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "./qwen05b-cbt-lora-adapter",
    torch_dtype=torch.float16,
    device_map="auto"
)

merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("./qwen05b-cbt-lora-merged")
tokenizer.save_pretrained("./qwen05b-cbt-lora-merged")

print("Merged model saved")

In [ ]:
def get_folder_size_mb(folder):
    total_size = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

print("Adapter size (MB):", round(get_folder_size_mb("./qwen05b-cbt-lora-adapter"), 2))
print("Merged model size (MB):", round(get_folder_size_mb("./qwen05b-cbt-lora-merged"), 2))

In [ ]:
import os
import shutil
from google.colab import files

bundle_dir = "/content/project_export"

os.makedirs(bundle_dir, exist_ok=True)

# Copier dossiers utiles
if os.path.exists("/content/qwen05b-cbt-lora-adapter"):
    shutil.copytree("/content/qwen05b-cbt-lora-adapter", f"{bundle_dir}/qwen05b-cbt-lora-adapter", dirs_exist_ok=True)

if os.path.exists("/content/qwen05b-cbt-lora-merged"):
    shutil.copytree("/content/qwen05b-cbt-lora-merged", f"{bundle_dir}/qwen05b-cbt-lora-merged", dirs_exist_ok=True)

if os.path.exists("/content/qwen05b-cbt-lora"):
    shutil.copytree("/content/qwen05b-cbt-lora", f"{bundle_dir}/qwen05b-cbt-lora", dirs_exist_ok=True)

# Copier fichiers utiles
files_to_copy = [
    "/content/final_dataset.csv",
    "/content/llm_chat_dataset (1).json"
]

for f in files_to_copy:
    if os.path.exists(f):
        shutil.copy(f, bundle_dir)

# Créer zip
shutil.make_archive("/content/cbt_project_all_files", "zip", bundle_dir)

# Télécharger
files.download("/content/cbt_project_all_files.zip")

# Test

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/qwen05b-cbt-lora-merged"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")

In [ ]:
def generate_response_clean(user_input):
    messages = [
        {"role": "system", "content": "You are a helpful and empathetic CBT assistant."},
        {"role": "user", "content": user_input}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 🔥 CLEAN OUTPUT (important)
    if "assistant" in decoded.lower():
        response = decoded.split("assistant")[-1]
    else:
        response = decoded

    return response.strip()

In [ ]:
response = generate_response("I feel really anxious about my future.")
print(response)

## Conclusion

In this notebook, we successfully fine-tuned a conversational language model for CBT-style dialogue using LoRA.

### What was achieved
- Loaded and prepared a chat-formatted dataset
- Applied parameter-efficient fine-tuning (LoRA)
- Trained the model using SFTTrainer in a Colab environment
- Merged the LoRA adapter into the base model
- Saved a standalone model ready for inference

### Key outcomes
- The model generates coherent and empathetic responses
- The final model remains lightweight (~1GB), suitable for deployment
- No external API is required for inference

### Limitations
- The model does not yet adapt responses based on user emotion
- No decision-making logic is applied to guide responses
- Responses are static (same behavior for all inputs)

### Next steps
- Add an emotion classification module
- Build a decision engine to adapt responses
- Implement dynamic prompt generation
- Deploy the model using a backend API (FastAPI)

This notebook establishes a solid foundation for building a full CBT conversational AI system.